# KTN Image Engine V1 — Colab FLUX.1-schnell FP8
Chạy ComfyUI + KTN Image Gateway trên GPU Colab. Khi chuyển sang GPU VPS, API `/v1/images/generations` giữ nguyên.

## 🚀 ONE-CLICK RESTORE — Image Engine + Render Worker\nSau khi Colab reset: chọn GPU rồi chỉ chạy **cell ngay bên dưới**. Cell sẽ clone repo và tự dựng lại FLUX/ComfyUI, KTN Image Gateway, MoneyPrinterTurbo Render Worker và 2 Cloudflare tunnel.\n

In [ ]:
# COLAB-BOOT-02 — ONE-CLICK FULL STACK RESTORE\nimport os, shutil, subprocess\nrepo='/content/KTN-AI-Video-Studio'\nbranch='ui-vn-01-vietnamese-baseline'\nif os.path.exists(repo):\n    shutil.rmtree(repo)\nsubprocess.run([\n    'git','clone','-q','-b',branch,\n    'https://github.com/jackylehoangle/KTN-AI-Video-Studio.git',repo\n], check=True)\nboot=os.path.join(repo,'colab','boot_all.py')\nexec(compile(open(boot,'r',encoding='utf-8').read(),boot,'exec'),globals(),globals())\n

In [ ]:
import os, subprocess, time, secrets, re, requests
subprocess.run(['nvidia-smi'])

In [ ]:
!rm -rf /content/KTN-AI-Video-Studio /content/ComfyUI
!git clone -q -b ui-vn-01-vietnamese-baseline https://github.com/jackylehoangle/KTN-AI-Video-Studio.git /content/KTN-AI-Video-Studio
!git clone -q https://github.com/Comfy-Org/ComfyUI.git /content/ComfyUI
!pip -q install -r /content/ComfyUI/requirements.txt
!pip -q install fastapi==0.136.3 uvicorn==0.32.1 requests==2.33.1 pydantic pillow
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('Cài đặt xong.')

In [ ]:
model='/content/ComfyUI/models/checkpoints/flux1-schnell-fp8.safetensors'
if not os.path.exists(model):
    !wget -c https://huggingface.co/Comfy-Org/flux1-schnell/resolve/main/flux1-schnell-fp8.safetensors -O "$model"
print('Model GB =', round(os.path.getsize(model)/1024**3,2))

In [ ]:
repo='/content/KTN-AI-Video-Studio'
token=secrets.token_urlsafe(32)
os.environ['COMFYUI_BASE_URL']='http://127.0.0.1:8188'
os.environ['COMFYUI_WORKFLOW_PATH']=repo+'/ktn_image_gateway/workflows/flux_schnell_api.json'
os.environ['KTN_IMAGE_GATEWAY_TOKEN']=token
comfy_log=open('/content/comfyui.log','w')
comfy=subprocess.Popen(['python','main.py','--listen','127.0.0.1','--port','8188','--lowvram'],cwd='/content/ComfyUI',stdout=comfy_log,stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        if requests.get('http://127.0.0.1:8188/system_stats',timeout=2).ok: break
    except Exception: pass
    time.sleep(1)
else: raise RuntimeError('ComfyUI không khởi động được. Xem /content/comfyui.log')
gateway_log=open('/content/ktn_gateway.log','w')
gateway=subprocess.Popen(['python','-m','uvicorn','ktn_image_gateway.main:app','--host','0.0.0.0','--port','8189'],cwd=repo,stdout=gateway_log,stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        if requests.get('http://127.0.0.1:8189/health',timeout=2).ok: break
    except Exception: pass
    time.sleep(1)
else: raise RuntimeError('KTN Image Gateway không khởi động được. Xem /content/ktn_gateway.log')
print(requests.get('http://127.0.0.1:8189/health').json())

In [ ]:
tunnel=subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--url','http://127.0.0.1:8189','--no-autoupdate'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
public_url=None
deadline=time.time()+60
while time.time()<deadline:
    line=tunnel.stdout.readline()
    if line:
        print(line.strip())
        m=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',line)
        if m:
            public_url=m.group(0); break
if not public_url: raise RuntimeError('Không lấy được Cloudflare Quick Tunnel URL')
print('\n=== COPY 2 BIẾN NÀY SANG VERCEL PREVIEW ===')
print('KTN_IMAGE_GATEWAY_URL='+public_url)
print('KTN_IMAGE_GATEWAY_TOKEN='+token)
print('===========================================')

In [ ]:
# Smoke test đúng 1 ảnh. Chỉ chạy khi đã sẵn sàng test GPU.
payload={'model':'flux','prompt':'cinematic sunrise over Ho Chi Minh City, realistic professional film still','size':'1360x768','n':1,'response_format':'b64_json'}
r=requests.post(public_url+'/v1/images/generations',json=payload,headers={'Authorization':'Bearer '+token},timeout=900)
print('HTTP',r.status_code)
data=r.json()
if r.ok:
    import base64, io
    from PIL import Image
    from IPython.display import display
    display(Image.open(io.BytesIO(base64.b64decode(data['data'][0]['b64_json']))))
else:
    print(data)